# DID Lab: 教学 — Angrist & Pischke MHE Chapter 4

> 教学笔记本, 对应 Angrist & Pischke (2009) *Mostly Harmless Econometrics* Chapter 4.
> 演示基本 2x2 DID + cluster SE + 平行趋势图。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

# Simulate MHE Chapter 4 example: Card-Krueger (1994) style minimum wage DID
# Treated: NJ (raised min wage in Feb 1992)
# Control: PA (no change)
# Outcome: Employment
rows = []
for store in range(50):  # 25 NJ + 25 PA
    is_treated = store < 25
    state = 'NJ' if is_treated else 'PA'
    for period in ['Nov1991', 'Feb1992']:
        post = int(period == 'Feb1992')
        # DGP: NJ has +2 employees post-treatment (Card-Krueger finding)
        base = 20 + np.random.normal(0, 3)
        treat_effect = 2.0 if (is_treated and post) else 0.0
        rows.append({
            'store_id': f'S{store:03d}',
            'state': state,
            'period': period,
            'post': post,
            'treated': int(is_treated),
            'did': int(is_treated and post),
            'employment': base + treat_effect,
        })
df = pd.DataFrame(rows)
print(f'Loaded {len(df)} obs, {df.store_id.nunique()} stores')
df.head()

In [ ]:
# Step 1: Group means (DID by hand)
means = df.groupby(['state', 'period'])['employment'].mean().unstack()
print('\n=== Group Means ===')
print(means)

did_by_hand = (means.loc['NJ', 'Feb1992'] - means.loc['NJ', 'Nov1991']) - \
              (means.loc['PA', 'Feb1992'] - means.loc['PA', 'Nov1991'])
print(f'\nDID by hand: {did_by_hand:.3f}')

In [ ]:
# Step 2: OLS regression with cluster-robust SE
import statsmodels.api as sm

X = sm.add_constant(df[['treated', 'post', 'did']])
y = df['employment']
model = sm.OLS(y, X).fit(cov_type='cluster', cov_kwds={'groups': df['store_id']})
print('\n=== OLS with cluster SE ===')
print(model.summary().tables[1])

In [ ]:
# Step 3: Visualize the DID
fig, ax = plt.subplots(figsize=(8, 5))
for state, color in [('NJ', 'C0'), ('PA', 'C1')]:
    sub = df[df.state == state].groupby('period')['employment'].mean()
    ax.plot(sub.index, sub.values, marker='o', linewidth=2, label=state, color=color)
ax.set_ylabel('Employment')
ax.set_title('Card-Krueger DID: NJ vs PA')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('output/mhe_ch4_did.png', dpi=300, bbox_inches='tight')
print('✓ Saved to output/mhe_ch4_did.png')

## 教学要点

1. **DID = (Y_treat_post - Y_treat_pre) - (Y_control_post - Y_control_pre)**
2. **标准误必须聚类**: Card-Krueger 样本只有 50 家店, 但每个店 2 期 → 必须 store-level cluster
3. **平行趋势假设**: 我们假设 PA 是 NJ 的反事实 (counterfactual)。若 NJ 和 PA 在 pre-period 趋势不同, DID 偏差。

进一步: `notebooks/02_iv_lab.ipynb` (IV), `notebooks/03_rdd_lab.ipynb` (RDD)